# Eksperyment: Surface-adjusted Elo (Sprint 5)

## Cel
Dodac do baseline rating **Elo** (jak w szachach) -- przewidujacy, aktualizowany wynikami meczow,
w przeciwienstwie do rankingu ATP (suma punktow do rozstawiania). Dwa ratingi: ogolny + per
nawierzchnia. Cztery cechy: `elo_diff`, `surface_elo_diff`, `elo_win_prob`, `surface_elo_win_prob`.

## Metoda
Elo liczony SAMODZIELNIE z danych Sackmanna -- z natury sekwencyjny (expanding window), wiec
leakage-safe. K-factor dynamiczny (FiveThirtyEight). **Walidacja walk-forward** przez kilka sezonow
+ test parowany McNemar (lekcja: pojedynczy test klamie).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

## Uruchomienie pelnej walidacji walk-forward (od zera)
Kazdy sezon: baseline + model z Elo na tych samych meczach, parowanie per-mecz.

In [2]:
import main_48_cech_elo as m
m.main()


===== ROK 2020 =====


  baseline=0.6176  +elo=0.6434  delta=+0.0257  (elo feat ranks /44: {'elo_diff': 3, 'surface_elo_diff': 1, 'elo_win_prob': 5, 'surface_elo_win_prob': 2})



===== ROK 2021 =====


  baseline=0.6667  +elo=0.6743  delta=+0.0077  (elo feat ranks /44: {'elo_diff': 2, 'surface_elo_diff': 5, 'elo_win_prob': 3, 'surface_elo_win_prob': 6})



===== ROK 2022 =====


...


===== ROK 2024 =====


  baseline=0.6186  +elo=0.6356  delta=+0.0169  (elo feat ranks /44: {'elo_diff': 3, 'surface_elo_diff': 6, 'elo_win_prob': 4, 'surface_elo_win_prob': 5})



===== ROK 2025 =====


  baseline=0.6566  +elo=0.6377  delta=-0.0189  (elo feat ranks /44: {'elo_diff': 1, 'surface_elo_diff': 6, 'elo_win_prob': 2, 'surface_elo_win_prob': 5})



WALK-FORWARD: baseline vs baseline + surface-adjusted ELO
 year   n  baseline    elo   delta
 2020 272    0.6176 0.6434  0.0257
 2021 522    0.6667 0.6743  0.0077
 2022 547    0.6728 0.6965  0.0238
 2023 561    0.6346 0.6328 -0.0018
 2024 590    0.6186 0.6356  0.0169
 2025 530    0.6566 0.6377 -0.0189
------------------------------------------------------------------------
POOLED (3022): baseline=0.6463  +elo=0.6539  delta=+0.0076
  delta dodatnia w 4/6 latach
  McNemar: b=119 c=142 z=1.36 p=0.1733
  => Brak istotnosci (p>=0.05).

Sredni rank cech Elo (im nizszy tym wazniejsze):
  elo_diff               sredni rank 2.2
  surface_elo_diff       sredni rank 4.7
  elo_win_prob           sredni rank 3.8
  surface_elo_win_prob   sredni rank 4.0


## Wnioski
Cechy Elo **dominuja waznosc** (elo_diff to top-2 cecha) -- model mocno ich uzywa. ALE pooled delta
jest mala (rzedu +0.5 p.p.) i **nieistotna** (McNemar p >> 0.05). Powod: Elo jest silnym, ale
REDUNDANTNYM sygnalem -- baseline ma juz ranking ATP, ktory mierzy to samo. Literaturowe "Elo ~70%"
dotyczy Elo jako GLOWNEGO sygnalu, nie dodatku do modelu z rankingiem. Elo zaplaciloby dopiero przy
duzo dluzszej historii do rozgrzewki ratingow.